# Feature Extraction Methods in NLP

This notebook continues from the previous text preprocessing notebook. Here, we use the cleaned and tokenized data to extract features for machine learning models. We'll cover Bag of Words, TF-IDF, and Word Embeddings, with simple explanations and code examples.

## 1. Import Required Libraries

Let's import the necessary Python libraries for feature extraction: scikit-learn for Bag of Words and TF-IDF, and gensim for Word2Vec embeddings.

In [1]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from gensim.models import Word2Vec
import numpy as np

## 2. Sample Corpus

We'll use a small sample corpus of English sentences. Next, we'll define a function to apply all the text preprocessing steps from the previous notebook to each sentence.

In [2]:
# Example corpus: a couple of English sentences
corpus = [
    "NLP is fun! Let's clean, tokenize, and analyze this text 123 times.",
    "Text preprocessing is essential for NLP tasks. Let's see how it works!"
]
print(corpus)

["NLP is fun! Let's clean, tokenize, and analyze this text 123 times.", "Text preprocessing is essential for NLP tasks. Let's see how it works!"]


### Text Preprocessing Function

Let's define a function that applies cleaning, lowercasing, tokenization, stop word removal, stemming, and lemmatization to a sentence, just like in the previous notebook.

In [ ]:
import nltk


nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

In [4]:
import re
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer


def preprocess_text(text):
    # Remove unwanted characters (punctuation)
    cleaned_text = re.sub(r'[^\w\s]', '', text)
    # Lowercase
    cleaned_text = cleaned_text.lower()
    # Tokenize
    tokens = word_tokenize(cleaned_text)
    # Remove stop words
    stop_words = set(stopwords.words('english'))
    tokens = [word for word in tokens if word not in stop_words]
    # Stemming
    stemmer = PorterStemmer()
    stemmed = [stemmer.stem(word) for word in tokens]
    # Lemmatization
    lemmatizer = WordNetLemmatizer()
    lemmatized = [lemmatizer.lemmatize(word) for word in tokens]
    return {
        'raw': text,
        'cleaned': cleaned_text,
        'tokens': tokens,
        'stemmed': " ".join(stemmed),
        'lemmatized': " ".join(lemmatized)
    }

# Apply to each sentence in the corpus
preprocessed = [preprocess_text(sentence) for sentence in corpus]
for i, result in enumerate(preprocessed):
    print(f"Sentence {i+1}:")
    print(result)
    print()

Sentence 1:
{'raw': "NLP is fun! Let's clean, tokenize, and analyze this text 123 times.", 'cleaned': 'nlp is fun lets clean tokenize and analyze this text 123 times', 'tokens': ['nlp', 'fun', 'lets', 'clean', 'tokenize', 'analyze', 'text', '123', 'times'], 'stemmed': 'nlp fun let clean token analyz text 123 time', 'lemmatized': 'nlp fun let clean tokenize analyze text 123 time'}

Sentence 2:
{'raw': "Text preprocessing is essential for NLP tasks. Let's see how it works!", 'cleaned': 'text preprocessing is essential for nlp tasks lets see how it works', 'tokens': ['text', 'preprocessing', 'essential', 'nlp', 'tasks', 'lets', 'see', 'works'], 'stemmed': 'text preprocess essenti nlp task let see work', 'lemmatized': 'text preprocessing essential nlp task let see work'}



## 3. Bag of Words (BoW)

Bag of Words represents text as a collection of word counts, ignoring grammar and word order. Each unique word becomes a feature. You can use your preprocessed tokens from the previous notebook or the raw corpus here.

In [6]:
sentences = [item['lemmatized'] for item in preprocessed]

vectorizer = CountVectorizer()
X_bow = vectorizer.fit_transform(sentences)
print("Bag of Words Array:\n", X_bow.toarray())
print("Feature Names:", vectorizer.get_feature_names_out())

Bag of Words Array:
 [[1 1 1 0 1 1 1 0 0 0 1 1 1 0]
 [0 0 0 1 0 1 1 1 1 1 1 0 0 1]]
Feature Names: ['123' 'analyze' 'clean' 'essential' 'fun' 'let' 'nlp' 'preprocessing'
 'see' 'task' 'text' 'time' 'tokenize' 'work']


## 4. TF-IDF (Term Frequency-Inverse Document Frequency)

TF-IDF weighs words based on their frequency in a document relative to their frequency across all documents, highlighting more relevant words. This step works best on cleaned text, so use your preprocessed corpus if available.

In [7]:
# TF-IDF also expects documents, so use the same joined string
tfidf_vectorizer = TfidfVectorizer()
X_tfidf = tfidf_vectorizer.fit_transform(sentences)
print("TF-IDF Array:\n", X_tfidf.toarray())
print("Feature Names:", tfidf_vectorizer.get_feature_names_out())

TF-IDF Array:
 [[0.36469323 0.36469323 0.36469323 0.         0.36469323 0.25948224
  0.25948224 0.         0.         0.         0.25948224 0.36469323
  0.36469323 0.        ]
 [0.         0.         0.         0.39166832 0.         0.27867523
  0.27867523 0.39166832 0.39166832 0.39166832 0.27867523 0.
  0.         0.39166832]]
Feature Names: ['123' 'analyze' 'clean' 'essential' 'fun' 'let' 'nlp' 'preprocessing'
 'see' 'task' 'text' 'time' 'tokenize' 'work']


## 5. Word Embeddings (Word2Vec)

Word embeddings represent words as dense vectors that capture semantic meaning. We'll use gensim's Word2Vec on a tokenized version of our sample corpus. You can also use your filtered tokens from the previous notebook for more realistic results.

In [11]:
# Import simple_preprocess for better tokenization
from gensim.utils import simple_preprocess

# Convert corpus to list of token lists for Word2Vec
corpora = [simple_preprocess(item['lemmatized']) for item in preprocessed]
corpora

[['nlp', 'fun', 'let', 'clean', 'tokenize', 'analyze', 'text', 'time'],
 ['text', 'preprocessing', 'essential', 'nlp', 'task', 'let', 'see', 'work']]

In [13]:
# For Word2Vec, use the tokens as a single sentence
model = Word2Vec(corpora, vector_size=10, window=5, min_count=1, workers=1, seed=42)
# Get the embedding for the word 'nlp'
print(model.wv.index_to_key)
vector = model.wv['nlp']
print("Embedding for 'nlp':\n", vector)

['text', 'let', 'nlp', 'work', 'see', 'task', 'essential', 'preprocessing', 'time', 'analyze', 'tokenize', 'clean', 'fun']
Embedding for 'nlp':
 [ 7.0381167e-05 -2.5840402e-02 -6.3490078e-02  8.5352995e-02
  5.6313492e-02  2.8773021e-02 -1.9517135e-02  6.4552322e-02
  9.0858219e-03 -1.1317169e-02]


In [15]:
model.wv.most_similar('work')

[('nlp', 0.42360222339630127),
 ('task', 0.26657161116600037),
 ('essential', 0.25709068775177),
 ('tokenize', 0.1051425114274025),
 ('preprocessing', 0.008240253664553165),
 ('fun', -0.0953390821814537),
 ('analyze', -0.11550464481115341),
 ('time', -0.12237732112407684),
 ('text', -0.42465025186538696),
 ('see', -0.4545871913433075)]

> Note: In practice, you would use a larger and more diverse corpus for training Word2Vec to get meaningful embeddings.